# Preprocessing Pipeline - Wilson et al. EEG RSA Replication

This notebook demonstrates the preprocessing pipeline for EEG data analysis.

## Pipeline Steps
1. Load raw data
2. Filter signals (bandpass)
3. Bad channel detection and interpolation
4. Re-referencing
5. Artifact rejection (ICA)
6. Epoching
7. Baseline correction
8. Epoch rejection
9. Save preprocessed data

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from mne.preprocessing import ICA
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Configuration
DATA_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True, parents=True)

print('Environment setup complete')

## 1. Load Raw Data

Load the raw EEG data and events.

In [ ]:
# TODO: Update with actual data loading
# raw = mne.io.read_raw_fif('path/to/raw.fif', preload=True)

# For demonstration, create sample data
info = mne.create_info(
    ch_names=['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2'],
    sfreq=250,
    ch_types='eeg'
)
data = np.random.randn(10, 50000)  # 10 channels, 200 seconds
raw = mne.io.RawArray(data, info)

# Set montage
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, on_missing='warn')

print(f'Loaded data: {raw}')

## 2. Bandpass Filtering

Apply bandpass filter to remove slow drifts and high-frequency noise.

Following Wilson et al., we use a [0.1, 40] Hz bandpass filter.

In [ ]:
# Filter parameters
l_freq = 0.1  # High-pass cutoff
h_freq = 40.0  # Low-pass cutoff

# Apply filter
raw_filt = raw.copy().filter(l_freq, h_freq, fir_design='firwin')

print(f'Applied {l_freq}-{h_freq} Hz bandpass filter')

In [ ]:
# Compare PSD before and after filtering
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

raw.compute_psd(fmax=50).plot(axes=axes[0], show=False)
axes[0].set_title('Before Filtering')

raw_filt.compute_psd(fmax=50).plot(axes=axes[1], show=False)
axes[1].set_title('After Filtering')

plt.tight_layout()

## 3. Bad Channel Detection

Identify and interpolate bad channels using automated detection.

In [ ]:
# Detect bad channels based on amplitude
# TODO: Implement more sophisticated bad channel detection

# For now, manually specify bad channels if known
raw_filt.info['bads'] = []  # e.g., ['Fp1', 'O2']

if raw_filt.info['bads']:
    print(f'Bad channels detected: {raw_filt.info["bads"]}')    # Interpolate bad channels
    raw_filt.interpolate_bads(reset_bads=True)
    print('Bad channels interpolated')
else:
    print('No bad channels detected')

## 4. Re-referencing

Apply average reference to reduce the impact of reference electrode choice.

In [ ]:
# Set average reference
raw_filt.set_eeg_reference('average', projection=True)
raw_filt.apply_proj()

print('Applied average reference')

## 5. Independent Component Analysis (ICA)

Apply ICA to identify and remove ocular and muscle artifacts.

In [ ]:
# Initialize ICA
n_components = 10  # Number of ICA components
ica = ICA(n_components=n_components, random_state=42, max_iter=800)

# Fit ICA
print('Fitting ICA...')
ica.fit(raw_filt)
print(f'ICA fitting complete. Found {ica.n_components_} components')

In [ ]:
# Plot ICA components
# ica.plot_components(inst=raw_filt)

# Plot component time courses
# ica.plot_sources(raw_filt)

In [ ]:
# Automatic artifact detection
# Detect EOG artifacts
eog_indices, eog_scores = ica.find_bads_eog(raw_filt, threshold=3.0)
print(f'EOG components detected: {eog_indices}')

# Detect ECG artifacts (if ECG channel available)
# ecg_indices, ecg_scores = ica.find_bads_ecg(raw_filt, threshold=3.0)
# print(f'ECG components detected: {ecg_indices}')

# Mark bad components
ica.exclude = eog_indices
print(f'\nExcluding ICA components: {ica.exclude}')

In [ ]:
# Apply ICA to remove artifacts
raw_clean = raw_filt.copy()
ica.apply(raw_clean)

print('ICA artifacts removed')

In [ ]:
# Compare before and after ICA
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Before ICA
data_before, times = raw_filt[0, :2500]
axes[0].plot(times, data_before.T * 1e6, linewidth=0.5)
axes[0].set_title('Before ICA')
axes[0].set_ylabel('Amplitude (µV)')

# After ICA
data_after, times = raw_clean[0, :2500]
axes[1].plot(times, data_after.T * 1e6, linewidth=0.5)
axes[1].set_title('After ICA')
axes[1].set_ylabel('Amplitude (µV)')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()

## 6. Epoching

Extract epochs around experimental events.

In [ ]:
# Extract events
# TODO: Update with actual event extraction
# events = mne.find_events(raw_clean, stim_channel='STI101')

# For demonstration
events = np.array([[i * 500, 0, j % 4 + 1] for i, j in enumerate(range(100))])

# Define event IDs
event_id = {
    'condition_1': 1,
    'condition_2': 2,
    'condition_3': 3,
    'condition_4': 4
}

print(f'Found {len(events)} events')

In [ ]:
# Epoch parameters
tmin, tmax = -0.2, 1.0  # Time window around events
baseline = (None, 0)  # Baseline period

# Create epochs
epochs = mne.Epochs(
    raw_clean,
    events,
    event_id,
    tmin,
    tmax,
    baseline=baseline,
    preload=True,
    reject=None  # Will apply rejection later
)

print(f'Created {len(epochs)} epochs')
print(f'Epoch duration: {tmin} to {tmax} seconds')

## 7. Epoch Rejection

Reject epochs with excessive artifacts based on amplitude thresholds.

In [ ]:
# Define rejection criteria (in Volts)
reject_criteria = dict(
    eeg=100e-6  # 100 µV
)

# Apply rejection
epochs.drop_bad(reject=reject_criteria)

print(f'Epochs after rejection: {len(epochs)}')
print(f'Rejection rate: {(1 - len(epochs) / len(events)) * 100:.1f}%')

In [ ]:
# Check epochs per condition
for condition, code in event_id.items():
    n_epochs = len(epochs[condition])
    print(f'{condition}: {n_epochs} epochs')

## 8. Visualize Preprocessed Data

Examine the cleaned and epoched data.

In [ ]:
# Plot average ERP for each condition
evoked_dict = {cond: epochs[cond].average() for cond in event_id.keys()}

# Plot ERPs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (cond, evoked) in enumerate(evoked_dict.items()):
    evoked.plot(axes=axes[idx], show=False, spatial_colors=True, gfp=True)
    axes[idx].set_title(f'{cond} (n={len(epochs[cond])})')

plt.tight_layout()

In [ ]:
# Plot topographic maps at key time points
times = np.arange(0.1, 0.6, 0.1)

for cond, evoked in evoked_dict.items():
    fig = evoked.plot_topomap(times, ch_type='eeg', time_unit='s')
    fig.suptitle(f'Topographic Maps - {cond}', fontsize=14)
    plt.tight_layout()

## 9. Save Preprocessed Data

Save cleaned epochs for subsequent analysis.

In [ ]:
# Save epochs
output_file = PROCESSED_DIR / 'preprocessed_epochs-epo.fif'
epochs.save(output_file, overwrite=True)

print(f'Saved preprocessed epochs to {output_file}')

In [ ]:
# Save preprocessing summary
summary = {
    'n_channels': len(epochs.ch_names),
    'sampling_rate': epochs.info['sfreq'],
    'n_epochs_total': len(epochs),
    'epoch_duration': f'{tmin} to {tmax} s',
    'baseline': str(baseline),
    'filter': f'{l_freq}-{h_freq} Hz',
    'ica_components_removed': ica.exclude,
    'bad_channels': raw_filt.info['bads']
}

# Add per-condition epoch counts
for cond in event_id.keys():
    summary[f'n_epochs_{cond}'] = len(epochs[cond])

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['Value']

print('\n=== Preprocessing Summary ===')
print(summary_df)

summary_df.to_csv(PROCESSED_DIR / 'preprocessing_summary.csv')

## Wilson et al. Specific Preprocessing Notes

### TODO: Document specific parameters from Wilson et al.

- Filtering: [specify exact filter parameters]
- Re-referencing: [specify reference type]
- ICA: [specify number of components and rejection criteria]
- Epoching: [specify time windows and baseline]
- Rejection: [specify amplitude thresholds]

### Quality Checks
- [ ] Verify all preprocessing steps match original study
- [ ] Check epoch counts are sufficient for RSA
- [ ] Confirm artifact removal is adequate
- [ ] Validate signal quality in preprocessed data

## Next Steps

1. ✓ Data successfully preprocessed
2. → Proceed to RSA analysis (see `03_rsa_analysis.ipynb`)
3. → Compute representational dissimilarity matrices
4. → Perform statistical tests
5. → Generate figures for publication